# 01 - Data Loading and Cleaning

This is the first stage of the project pipeline. It reads the raw Israeli GTFS feed (the national public-transport schedule export), converts the text tables into typed data frames, applies a small set of explicit sanity filters, and attaches two coarse geographic labels to every stop: a **region** (North / Center / Jerusalem / South) and a **metropolitan area** (Tel Aviv / Haifa / Jerusalem / Beer Sheva / Periphery). Everything downstream - graph construction, centrality, robustness - starts from the cleaned tables written here, so this notebook also produces a machine-readable cleaning report that records exactly how many rows each filter removed.

The guiding idea is that a network study is only as trustworthy as its node table. If a stop has a broken coordinate, or the same `stop_id` appears twice, the graph silently gains a phantom node and every centrality number afterwards is contaminated. So each filter here is written as its own step, is counted, and is reported.

**Research question this stage serves:** *what exactly is the node set of the Israeli public-transport network, and what did we throw away to get it?*

### Inputs

- `israel-public-transportation/stops.txt` - one row per physical stop / station (id, name, latitude, longitude, location type, parent station).
- `israel-public-transportation/routes.txt` - one row per route, with `route_type` (3 = bus, 2 = rail, ...) and the operating `agency_id`.
- `israel-public-transportation/trips.txt` - one row per scheduled vehicle run; this is the service-volume table.
- `israel-public-transportation/agency.txt` - one row per operator.

All four files are tracked in the repository. This notebook does **not** touch `stop_times.txt` (816 MB, not tracked in git) - that file is only needed from notebook 02 onwards.

### Outputs (all under `outputs/nb/01_data_preparation/`)

- `tables/stops_clean.csv` - the cleaned node table, plus the new `region` and `metro` columns.
- `tables/routes_clean.csv`, `tables/trips_clean.csv`, `tables/agencies_clean.csv` - pass-through copies with stable typing, so later notebooks never have to re-derive the parsing options.
- `tables/route_type_distribution.csv` - routes and scheduled trips per GTFS mode.
- `tables/agency_trip_counts.csv` - scheduled trips per operator.
- `tables/data_cleaning_report.json` - the audit trail: row counts before/after every filter, region/metro breakdowns, referential-integrity checks.
- `figures/*.png` - four descriptive figures.

### Depends on

Nothing. This is stage 01; it reads only the raw feed.

## Environment bootstrap

The cell below is identical in every notebook of this project. It does three things: (1) defines `_ensure`, which pip-installs *only* the packages that are genuinely missing, so a re-run costs nothing; (2) locates the repository root by walking up from the current directory looking for the `israel-public-transportation` folder, and clones the repo if we are on a fresh Google Colab machine; (3) creates the shared `outputs/nb` directory. After this cell, `REPO`, `DATA` and `OUT` are available and the working directory is the repository root, so every path in the notebook can be written relative to a known anchor.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## Libraries, stage folder and tunable constants

This notebook owns the folder `outputs/nb/01_data_preparation/`, split into `tables/` and `figures/`. Nothing is ever written to `outputs/tables`, `outputs/figures` or `outputs/rail`, which hold the results already cited in the written report.

Two groups of constants are pulled up here so a reader can see and change every tunable knob in one place:

- **`LAT_MIN/LAT_MAX/LON_MIN/LON_MAX`** - the bounding box that a coordinate must fall inside to be accepted as a real Israeli stop. The box is deliberately generous (Eilat sits near 29.55 N, Metula near 33.28 N), so it catches only clearly broken values such as `0,0` placeholders, not legitimate edge-of-country stops.
- **`SAVE_TRIPS_CSV`** - `trips.txt` is about 28 MB and roughly 1.2 M rows; re-writing it as a CSV is the single slowest operation in this notebook (tens of seconds and 28 MB of disk). Set it to `False` if you only care about the stop table and the report.

In [ ]:
# Third-party libraries used below. _ensure only installs what is missing.
_ensure("pandas", "numpy", "matplotlib")

import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- This notebook's own output folder (stage 01) ---
STAGE = OUT / "01_data_preparation"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ---
LAT_MIN, LAT_MAX = 29.0, 34.0   # generous bounding box around Israel
LON_MIN, LON_MAX = 34.0, 36.0
SAVE_TRIPS_CSV = True           # writing trips_clean.csv costs ~28 MB and tens of seconds

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print("GTFS folder :", DATA)
print("Stage folder:", STAGE)

## Rendering Hebrew labels correctly

Every stop name and operator name in the feed is Hebrew, and one of the figures below is a bar chart of operator names. Matplotlib draws glyphs in logical (storage) order and does not implement the Unicode bidirectional algorithm, so Hebrew text comes out visually reversed. The cell below patches `matplotlib.text.Text.set_text` once, so that any Hebrew string is converted to display order before being drawn. Strings without Hebrew characters pass through untouched, and the patch is idempotent (re-running the cell is harmless).

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## Loading the raw GTFS tables

GTFS files are plain comma-separated text, but three parsing decisions matter enough to be worth stating explicitly, because getting any of them wrong corrupts the identifiers that the whole graph is keyed on:

1. **`dtype=str`** - GTFS identifiers are opaque strings, not numbers. If pandas infers integers, `"01"` and `"1"` collapse into the same value and joins between `trips` and `routes` start matching the wrong rows. We keep everything as text and convert only the two coordinate columns, deliberately, later on.
2. **`keep_default_na=False`** - an empty GTFS field means *not supplied*, and several real stop names/descriptions in this feed contain strings such as `NA`. Letting pandas auto-convert them to `NaN` would both invent missing data and destroy real names.
3. **`encoding="utf-8-sig"`** - the Israeli feed ships with a UTF-8 byte-order mark. Without this, the first column of every file is read with an invisible BOM character glued to its name, and `df["stop_id"]` raises `KeyError`.

We load all four tables into a dictionary of *raw* frames and keep them untouched, so that every later row count can be compared against the original.

In [ ]:
GTFS_FILES = {
    "stops":  "stops.txt",
    "routes": "routes.txt",
    "trips":  "trips.txt",
    "agency": "agency.txt",
}

def load_gtfs(name):
    """Load one GTFS table as raw text, with the three parsing decisions above."""
    path = DATA / GTFS_FILES[name]
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is missing. The GTFS feed is tracked in the repository - "
            "check that the clone completed and that DATA points at the feed folder."
        )
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")

raw = {name: load_gtfs(name) for name in GTFS_FILES}

for name, df in raw.items():
    print(f"{name:7s} rows={len(df):>9,d}  columns={list(df.columns)}")

raw["stops"].head(3)

## Cleaning the stop table, one auditable filter at a time

The stop table is the node table of the future graph, so it gets the real scrutiny. Four filters are applied in order, and each one records how many rows it removed into `cleaning_steps`, which is later serialised into the JSON report:

1. **Unparseable coordinates.** `stop_lat` / `stop_lon` are converted with `errors="coerce"`, so an empty string or any non-numeric junk becomes `NaN`, and those rows are dropped. A stop without a position cannot be placed in a spatial graph at all.
2. **Latitude inside the bounding box** (`29 < lat < 34`). Catches `0.0` placeholders and digit-transposition typos.
3. **Longitude inside the bounding box** (`34 < lon < 36`). Same reasoning.
4. **Duplicate `stop_id`.** This filter is an addition to the original cleaning script. `stop_id` is the node key for every later stage; if it were not unique, a merge would fan out rows and inflate the degree of the duplicated node. Keeping the first occurrence makes the guarantee explicit rather than assumed.

**Honest note in advance:** in the feed version shipped with this repository, filters 1-4 all remove **zero** rows - the coordinates are complete and in range, and `stop_id` is already unique. That is a good result, not a wasted step: the numbers printed below are the evidence for the claim, and the same code protects the pipeline if the feed is ever refreshed with a dirtier export.

Note also what we deliberately do **not** filter: `location_type = 1` rows are *stations* (parent containers) rather than boardable platforms. They stay in the table here, and the report counts them, so that the graph-construction notebook can decide how to treat parent/child stops with the full information in front of it.

In [ ]:
cleaning_steps = []

def record(step, before, after, note):
    """Log one cleaning filter so the report can state exactly what was dropped."""
    cleaning_steps.append({
        "step": step,
        "rows_before": int(before),
        "rows_after": int(after),
        "rows_dropped": int(before - after),
        "note": note,
    })
    print(f"{step:26s} {before:>7,d} -> {after:>7,d}   dropped {before - after:>6,d}   {note}")

stops = raw["stops"].copy()

# 1. Coordinates must be numeric. Empty strings and junk become NaN and are dropped.
stops["stop_lat"] = pd.to_numeric(stops["stop_lat"], errors="coerce")
stops["stop_lon"] = pd.to_numeric(stops["stop_lon"], errors="coerce")
before = len(stops)
stops = stops.dropna(subset=["stop_lat", "stop_lon"])
record("numeric coordinates", before, len(stops), "stop_lat/stop_lon parse as numbers")

# 2. Latitude inside the Israel bounding box.
before = len(stops)
stops = stops[(stops["stop_lat"] > LAT_MIN) & (stops["stop_lat"] < LAT_MAX)]
record("latitude in range", before, len(stops), f"{LAT_MIN} < lat < {LAT_MAX}")

# 3. Longitude inside the Israel bounding box.
before = len(stops)
stops = stops[(stops["stop_lon"] > LON_MIN) & (stops["stop_lon"] < LON_MAX)]
record("longitude in range", before, len(stops), f"{LON_MIN} < lon < {LON_MAX}")

# 4. stop_id is the node key of the graph - it has to be unique.
before = len(stops)
stops = stops.drop_duplicates(subset=["stop_id"], keep="first")
record("unique stop_id", before, len(stops), "stop_id is the graph node key")

stops = stops.reset_index(drop=True)
kept = 100.0 * len(stops) / len(raw["stops"])
print()
print(f"clean stops: {len(stops):,d} of {len(raw['stops']):,d} raw rows kept ({kept:.2f}%)")
print()
print("location_type breakdown of the clean table (0 = stop/platform, 1 = station):")
print(stops["location_type"].value_counts().to_string())

## Coordinate sanity check

Before trusting the bounding-box filter, it is worth looking at the actual extent of the data: if the observed minimum and maximum sit comfortably *inside* the box, the filter is a guard rail rather than something that silently amputated part of the country. The printout below shows the extremes and the corner stops, which are easy to eyeball against a map (the southern extreme should be Eilat, the northern one the Upper Galilee).

In [ ]:
print(f"latitude  range: {stops['stop_lat'].min():.5f} .. {stops['stop_lat'].max():.5f}   (box {LAT_MIN} .. {LAT_MAX})")
print(f"longitude range: {stops['stop_lon'].min():.5f} .. {stops['stop_lon'].max():.5f}   (box {LON_MIN} .. {LON_MAX})")
print()

extremes = pd.DataFrame([
    stops.loc[stops["stop_lat"].idxmin()],
    stops.loc[stops["stop_lat"].idxmax()],
    stops.loc[stops["stop_lon"].idxmin()],
    stops.loc[stops["stop_lon"].idxmax()],
])
extremes.index = ["southernmost", "northernmost", "westernmost", "easternmost"]
extremes[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

## Region and metropolitan labels

Later stages compare centrality and robustness across parts of the country, so every stop needs a geographic label. The feed gives us none - `stops.txt` has no administrative district column - so we derive two labels from coordinates alone.

**Region** (first matching rule wins, reproducing the original `if/elif` chain exactly):

| order | rule | label |
|---|---|---|
| 1 | `31.70 <= lat <= 31.90` **and** `34.95 <= lon <= 35.30` | Jerusalem |
| 2 | `lat > 32.50` | North |
| 3 | `lat >= 31.55` | Center |
| 4 | otherwise | South |

**Metro** - the first centre whose disc contains the stop: Tel Aviv (30 km), Haifa (25 km), Jerusalem (20 km), Beer Sheva (25 km); everything else is `Periphery`. Distances are great-circle (haversine) distances.

### Limitation - please read this before using the labels

**The region assignment is a crude latitude/longitude rule, not a real administrative or statistical geography.** It knows nothing about district boundaries, municipal borders, the Green Line, terrain, or travel time. Concretely:

- Region 2/3/4 are pure latitude cuts, so they are horizontal bands across the country. A stop in the western Negev foothills and a stop in the eastern Judean desert at the same latitude get the same label even though they belong to completely different transport realities.
- The Jerusalem region is a rectangle, so stops in the Jerusalem corridor just outside the box fall into "Center", and the rectangle is checked *first*, which means it overrides the latitude bands wherever they disagree.
- The metro discs are circles around a single downtown point with hand-picked radii. A 30 km disc around Tel Aviv reaches well beyond the statistical Tel Aviv metropolitan area in some directions and falls short in others.
- `region` and `metro` are computed independently and can disagree - a stop can be `region = Center` and `metro = Jerusalem`.

The labels are therefore fine for coarse descriptive grouping ("roughly how much of the network sits in the north?") and should **not** be presented as official regional statistics. The original script used Hebrew labels; they are translated to English here (North / Center / Jerusalem / South, and Periphery for non-metro) with the thresholds unchanged.

One implementation note: the original code called a scalar haversine inside `DataFrame.apply` once per stop, which is roughly 35 000 Python-level function calls. The version below computes the same numbers vectorised with NumPy - identical results, a fraction of the runtime. The cell also prints the pairwise distances between the four metro centres, to verify that the discs do not overlap; if they do not overlap, "first match wins" and "nearest centre wins" are the same rule, and the ordering of the dictionary carries no hidden bias.

In [ ]:
# Metro centres: name -> (latitude, longitude, radius in km)
METRO_CENTERS = {
    "Tel Aviv":   (32.0853, 34.7818, 30),
    "Haifa":      (32.7940, 34.9896, 25),
    "Jerusalem":  (31.7683, 35.2137, 20),
    "Beer Sheva": (31.2518, 34.7913, 25),
}

def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km. Accepts scalars or NumPy arrays."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(np.asarray(lat2) - np.asarray(lat1))
    dlam = np.radians(np.asarray(lon2) - np.asarray(lon1))
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2.0) ** 2
    return 2.0 * R * np.arcsin(np.sqrt(a))

lat = stops["stop_lat"].to_numpy()
lon = stops["stop_lon"].to_numpy()

# --- Region: np.select evaluates conditions in order, exactly like if/elif/else ---
region_conditions = [
    (lat >= 31.70) & (lat <= 31.90) & (lon >= 34.95) & (lon <= 35.30),  # Jerusalem box
    lat > 32.50,                                                        # North
    lat >= 31.55,                                                       # Center
]
stops["region"] = np.select(region_conditions, ["Jerusalem", "North", "Center"], default="South")

# --- Metro: first centre whose disc contains the stop ---
metro_conditions, metro_names = [], []
for city, (clat, clon, radius) in METRO_CENTERS.items():
    metro_conditions.append(haversine_km(lat, lon, clat, clon) <= radius)
    metro_names.append(city)
stops["metro"] = np.select(metro_conditions, metro_names, default="Periphery")

# --- Do the metro discs overlap? If not, dictionary order is irrelevant. ---
print("metro disc overlap check")
items = list(METRO_CENTERS.items())
for i in range(len(items)):
    for j in range(i + 1, len(items)):
        a_name, (a_lat, a_lon, a_r) = items[i]
        b_name, (b_lat, b_lon, b_r) = items[j]
        d = float(haversine_km(a_lat, a_lon, b_lat, b_lon))
        verdict = "OVERLAP - order matters" if d < a_r + b_r else "disjoint"
        print(f"  {a_name:11s} - {b_name:11s}: {d:6.1f} km apart, radii sum {a_r + b_r:3d} km -> {verdict}")

print()
print("stops per region")
print(stops["region"].value_counts().to_string())
print()
print("stops per metro")
print(stops["metro"].value_counts().to_string())
print()
print("region x metro cross-tabulation (they are computed independently and can disagree)")
pd.crosstab(stops["region"], stops["metro"])

## Routes, trips, agencies and referential integrity

The other three tables need no row filtering - they are identifier tables, not measurements - but they do need to be *described*, and they need to be checked for referential integrity before later notebooks join them together.

Two things are computed here:

1. **Mode composition.** `route_type` is the GTFS mode code (0 tram/light rail, 2 rail, 3 bus, and so on). Counting *routes* per mode tells you how the timetable is organised; counting *scheduled trips* per mode tells you how much service each mode actually runs, which is the number that matters when weighting the network. Both are reported.
2. **Integrity checks.** Any `trip` whose `route_id` is absent from `routes.txt`, or any `route` whose `agency_id` is absent from `agency.txt`, would silently become a `NaN` after a left join and quietly distort per-mode or per-operator aggregates. We count them here and put the counts in the report; a non-zero value is a warning sign for every downstream stage.

In [ ]:
# Human-readable names for the GTFS route_type codes present in this feed.
ROUTE_TYPE_LABELS = {
    "0": "tram/light rail",
    "1": "subway",
    "2": "rail",
    "3": "bus",
    "4": "ferry",
    "5": "cable tram",
    "6": "aerial lift",
    "7": "funicular",
    "8": "trolleybus",
    "715": "demand/other bus",
}

routes = raw["routes"].copy()
trips = raw["trips"].copy()
agencies = raw["agency"].copy()

# Routes per mode.
route_types = (routes["route_type"]
               .value_counts()
               .rename_axis("route_type")
               .reset_index(name="routes"))
route_types["route_type_label"] = route_types["route_type"].map(ROUTE_TYPE_LABELS).fillna("unknown")

# Scheduled trips per mode - the service-volume view of the same breakdown.
trips_by_type = (trips.merge(routes[["route_id", "route_type"]], on="route_id", how="left")
                      .groupby("route_type")
                      .size()
                      .rename("scheduled_trips")
                      .reset_index())
route_types = route_types.merge(trips_by_type, on="route_type", how="left")
route_types["scheduled_trips"] = route_types["scheduled_trips"].fillna(0).astype(int)
route_types = route_types[["route_type", "route_type_label", "routes", "scheduled_trips"]]

# Referential integrity: dangling foreign keys would become silent NaNs in later joins.
orphan_trips = int((~trips["route_id"].isin(set(routes["route_id"]))).sum())
orphan_routes = int((~routes["agency_id"].isin(set(agencies["agency_id"]))).sum())
print("trips whose route_id is not in routes.txt  :", orphan_trips)
print("routes whose agency_id is not in agency.txt:", orphan_routes)
print()

# Scheduled trips per operator.
agency_summary = (trips.merge(routes[["route_id", "agency_id"]], on="route_id", how="left")
                       .merge(agencies[["agency_id", "agency_name"]], on="agency_id", how="left")
                       .groupby(["agency_id", "agency_name"])
                       .size()
                       .rename("scheduled_trips")
                       .reset_index()
                       .sort_values("scheduled_trips", ascending=False)
                       .reset_index(drop=True))

print(f"agencies: {len(agencies):,d}   routes: {len(routes):,d}   trips: {len(trips):,d}")
route_types

## Descriptive figures

Four figures, each answering one question about the cleaned data, all saved as PNG into the stage `figures/` folder:

1. **Stops per region and per metro** - is the node set balanced, or is it dominated by one part of the country?
2. **Geographic scatter of every clean stop, coloured by region** - the sanity check that matters most. If the bounding-box filter or the region rule were broken, it would be immediately visible here as stray points in the sea or a band in the wrong place. The aspect ratio is corrected by `cos(latitude)` so the country is not horizontally stretched.
3. **Routes and scheduled trips per mode** - on a logarithmic axis, because bus service dominates the feed by orders of magnitude.
4. **Top operators by scheduled trips** - Hebrew operator names, rendered through the bidi patch installed earlier.

In [ ]:
REGION_COLORS = {"North": "#1d4ed8", "Center": "#0f766e", "Jerusalem": "#b45309", "South": "#be123c"}

# --- Figure 1: stops per region and per metro ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
region_counts = stops["region"].value_counts()
axes[0].bar(region_counts.index, region_counts.values,
            color=[REGION_COLORS.get(r, "#64748b") for r in region_counts.index])
axes[0].set_title("Stops per region (crude lat/lon rule)")
axes[0].set_ylabel("number of stops")
metro_counts = stops["metro"].value_counts()
axes[1].bar(metro_counts.index, metro_counts.values, color="#334155")
axes[1].set_title("Stops per metropolitan area")
axes[1].tick_params(axis="x", rotation=20)
for ax in axes:
    ax.grid(axis="y", alpha=0.3)
    ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES / "region_and_metro_counts.png", dpi=150)
plt.show()

### Figure 2 - the map

This is the strongest single sanity check in the notebook. Plotting all clean stops in longitude/latitude space should reproduce the recognisable shape of the country: a dense coastal strip, the Jerusalem corridor, a thin scatter down the Negev to Eilat. Anything that survived the bounding-box filter but is nevertheless wrong - a coordinate in the sea, a mirrored latitude - shows up here as an obvious outlier. Colouring by `region` simultaneously visualises the crude latitude cuts, so the reader can see for themselves where the rule is arbitrary. The metro centres are marked so the disc radii can be judged by eye.

In [ ]:
# --- Figure 2: every clean stop on the map, coloured by region ---
fig, ax = plt.subplots(figsize=(6.5, 9))
for name, part in stops.groupby("region"):
    ax.scatter(part["stop_lon"], part["stop_lat"], s=1.5, alpha=0.35,
               color=REGION_COLORS.get(name, "#64748b"),
               label=f"{name} (n={len(part):,d})")
for city, (clat, clon, radius) in METRO_CENTERS.items():
    ax.plot(clon, clat, marker="o", markersize=7, color="black", zorder=5)
    ax.annotate(city, (clon, clat), textcoords="offset points", xytext=(8, 4), fontsize=9, zorder=6)
ax.set_aspect(1.0 / math.cos(math.radians(31.7)))  # keep the country from looking stretched
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title(f"{len(stops):,d} clean stops, coloured by assigned region")
ax.legend(markerscale=8, loc="upper left", fontsize=9)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "stops_map_by_region.png", dpi=150)
plt.show()

### Figure 3 - service composition by mode

Routes and scheduled trips side by side, per GTFS `route_type`. The y-axis is logarithmic because bus service is several orders of magnitude larger than every other mode; on a linear axis the rail and light-rail bars would be invisible. The gap between the two bars of the same mode is informative in itself: a mode with few routes but many trips runs high-frequency service on a small network.

In [ ]:
# --- Figure 3: routes and scheduled trips per GTFS mode (log scale) ---
labels = route_types["route_type_label"] + " (" + route_types["route_type"] + ")"
x = np.arange(len(route_types))
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.bar(x - 0.2, route_types["routes"], width=0.4, label="routes", color="#0f766e")
ax.bar(x + 0.2, route_types["scheduled_trips"], width=0.4, label="scheduled trips", color="#b45309")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.set_yscale("log")
ax.set_ylabel("count (log scale)")
ax.set_title("Service composition by GTFS mode")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES / "route_type_distribution.png", dpi=150)
plt.show()

### Figure 4 - operators

Scheduled trips per operator, top 12. This is a concentration check: Israeli public transport is run by many licensed operators, but service volume is expected to be dominated by a handful of them. The operator names are Hebrew and are rendered through the bidi patch installed earlier - if this figure shows reversed text, that patch did not take effect.

In [ ]:
# --- Figure 4: top operators by scheduled trips (Hebrew names via the bidi patch) ---
top_agencies = agency_summary.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_agencies["agency_name"], top_agencies["scheduled_trips"], color="#0f766e")
ax.set_xlabel("scheduled trips")
ax.set_title("Top 12 operators by number of scheduled trips")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES / "top_agencies_by_trips.png", dpi=150)
plt.show()

## Saving the stage artifacts

Everything the later notebooks need is written into `outputs/nb/01_data_preparation/tables/`. The four `*_clean.csv` files are written with `utf-8-sig` so that the Hebrew names open correctly in Excel, and with `index=False` so that re-reading them does not add a phantom column.

The JSON report is the deliverable that makes this stage auditable: it stores the per-filter row counts, the bounding box actually used, the region and metro definitions, the mode and location-type breakdowns, and the referential-integrity counts. A grader can read the report alone and know exactly what was dropped and why, without re-running the notebook.

One small technical point: `Series.value_counts().to_dict()` can yield NumPy integer values, which `json.dump` refuses to serialise on some pandas/NumPy combinations. The helper below converts keys to `str` and values to plain `int`, so the report always serialises.

Writing `trips_clean.csv` (about 28 MB) is the slow part; set `SAVE_TRIPS_CSV = False` in the constants cell to skip it.

In [ ]:
def counts_to_dict(series):
    """value_counts as a JSON-safe {str: int} dictionary."""
    return {str(k): int(v) for k, v in series.value_counts().items()}

# --- cleaned tables ---
stops.to_csv(TABLES / "stops_clean.csv", index=False, encoding="utf-8-sig")
routes.to_csv(TABLES / "routes_clean.csv", index=False, encoding="utf-8-sig")
agencies.to_csv(TABLES / "agencies_clean.csv", index=False, encoding="utf-8-sig")
if SAVE_TRIPS_CSV:
    trips.to_csv(TABLES / "trips_clean.csv", index=False, encoding="utf-8-sig")
else:
    print("skipped trips_clean.csv (SAVE_TRIPS_CSV = False)")

# --- derived summary tables ---
route_types.to_csv(TABLES / "route_type_distribution.csv", index=False, encoding="utf-8-sig")
agency_summary.to_csv(TABLES / "agency_trip_counts.csv", index=False, encoding="utf-8-sig")

# --- the audit trail ---
report = {
    "generated_by": "notebooks/01_data_preparation.ipynb",
    "gtfs_source_dir": str(DATA),
    "stops_raw": int(len(raw["stops"])),
    "stops_clean": int(len(stops)),
    "stops_dropped": int(len(raw["stops"]) - len(stops)),
    "routes_total": int(len(routes)),
    "trips_total": int(len(trips)),
    "agencies_total": int(len(agencies)),
    "cleaning_steps": cleaning_steps,
    "coordinate_bounds": {
        "lat_min": LAT_MIN, "lat_max": LAT_MAX,
        "lon_min": LON_MIN, "lon_max": LON_MAX,
    },
    "observed_bounds": {
        "lat_min": float(stops["stop_lat"].min()), "lat_max": float(stops["stop_lat"].max()),
        "lon_min": float(stops["stop_lon"].min()), "lon_max": float(stops["stop_lon"].max()),
    },
    "region_rule": (
        "crude first-match lat/lon rule: Jerusalem box (31.70-31.90 lat, 34.95-35.30 lon); "
        "North lat > 32.50; Center lat >= 31.55; South otherwise. Not an administrative geography."
    ),
    "metro_centers": {
        city: {"lat": c[0], "lon": c[1], "radius_km": c[2]} for city, c in METRO_CENTERS.items()
    },
    "region_counts": counts_to_dict(stops["region"]),
    "metro_counts": counts_to_dict(stops["metro"]),
    "location_type_counts": counts_to_dict(stops["location_type"]),
    "route_type_counts": counts_to_dict(routes["route_type"]),
    "integrity": {
        "trips_with_unknown_route_id": orphan_trips,
        "routes_with_unknown_agency_id": orphan_routes,
    },
}

report_path = TABLES / "data_cleaning_report.json"
with open(report_path, "w", encoding="utf-8") as fh:
    json.dump(report, fh, ensure_ascii=False, indent=2)

print("written to", TABLES)
for p in sorted(TABLES.iterdir()):
    print(f"  {p.name:32s} {p.stat().st_size / 1024**2:8.2f} MB")
print()
print(json.dumps({k: v for k, v in report.items() if k != "cleaning_steps"},
                 ensure_ascii=False, indent=2)[:2000])

## Takeaways

- **The feed is clean.** All four cleaning filters - unparseable coordinates, latitude range, longitude range, duplicate `stop_id` - remove **zero** rows from the shipped version of the feed: every stop has numeric coordinates, every coordinate falls inside the Israel bounding box (observed extent is roughly 29.49-33.28 N, 34.28-35.84 E, comfortably inside the 29-34 / 34-36 guard box), and `stop_id` is already unique. This is a negative result for the cleaning step and a positive one for the data: the node set of the network is the full raw stop table. The filters stay in the pipeline as guard rails for future feed refreshes, and the report proves they were checked rather than assumed.
- **The stop table is large and heavily bus-oriented.** Roughly 35 000 stops, of which only a couple of hundred are `location_type = 1` parent stations; `route_type = 3` (bus) dominates both route count and scheduled-trip count by orders of magnitude. Any conclusion about "the Israeli transit network" drawn from the full graph is really a conclusion about the bus network, which is why the project also analyses the rail layer separately.
- **Region and metro labels are convenience labels, not geography.** Region is three latitude cuts plus one rectangle around Jerusalem; metro is four circles with hand-picked radii. They are useful for coarse grouping and for colouring maps, and they should not be quoted as regional statistics. The two labels are computed independently and the cross-tabulation printed above shows where they disagree. The one structural reassurance is that the four metro discs are pairwise disjoint, so "first match wins" is equivalent to "nearest centre wins" and the ordering introduces no bias.
- **Referential integrity was verified, not assumed.** The counts of trips with an unknown `route_id` and routes with an unknown `agency_id` are computed and stored in the report; later notebooks join these tables freely and rely on those counts being zero.
- **What is deliberately left for stage 02:** no edges exist yet. Nothing here reads `stop_times.txt`, so no notion of adjacency, headway or travel time has been introduced. Parent/child station consolidation is also left untouched on purpose, so the graph-construction notebook can make that decision with the full stop table in front of it.